# Live Demo — The Blind Spots in Fraud Models

Companion notebook for the webinar. Run cells in order during the talk — each section header below maps directly to a slide in the deck, so you always know which cell to run next.

**Part A (Section 02 — The Standard Fix):** imbalanced classes, three fixes, simulated 10,000-loan dataset — mirrors the *Imbalanced Classes* tutorial exactly.

**Part B (Sections 04-06 — A Different Question / The Payoff / Unusual != Risky):** unsupervised anomaly detection on the real 24,977-loan LendingClub sample — mirrors the *Anomaly Detection* tutorial exactly.

These are two different datasets, deliberately — simulated data for the imbalance story (clean, controllable 94:6 split), real LendingClub data for the anomaly detection story (real borrower profiles worth inspecting). Say so explicitly when you switch between them in Part B.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Part A — Section 02: The Standard Fix

*Slide cue: run this right after the “10,000 Loan Applicants” slide, before the “94% / 0%” reveal.*

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, roc_auc_score, f1_score,
    recall_score, precision_score
)
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

X, y = make_classification(
    n_samples=10_000,
    n_features=15,
    n_informative=8,
    weights=[0.94, 0.06],   # 94% good loans, 6% defaults
    flip_y=0.01,
    random_state=42
)

feature_names = [
    "credit_score", "annual_income", "loan_amount", "loan_term",
    "debt_to_income", "num_open_accounts", "num_late_payments",
    "employment_length", "home_ownership", "loan_purpose",
    "interest_rate", "installment", "revolving_balance",
    "revolving_utilization", "total_accounts"
]
X = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {y_train.sum()} defaults / {len(y_train)} total "
      f"({y_train.mean()*100:.1f}% default rate)")

In [ ]:
def evaluate(name, y_true, y_pred, y_prob=None):
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred,
          target_names=["Good loan", "Default"]))
    if y_prob is not None:
        print(f"  AUC-ROC: {roc_auc_score(y_true, y_prob):.3f}")

### Baseline — the trap in action

*Slide cue: the “94% accuracy, 0% recall” reveal.*

In [ ]:
baseline = GradientBoostingClassifier(random_state=42)
baseline.fit(X_train, y_train)

evaluate(
    "Baseline — no imbalance handling",
    y_test,
    baseline.predict(X_test),
    baseline.predict_proba(X_test)[:, 1]
)

### Fix 1 — class weights

*Slide cue: “This Is a One-Line Fix” code slide.*

In [ ]:
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

weighted_model = GradientBoostingClassifier(random_state=42)
weighted_model.fit(X_train, y_train, sample_weight=sample_weights)

evaluate(
    "Fix 1 — class weights",
    y_test,
    weighted_model.predict(X_test),
    weighted_model.predict_proba(X_test)[:, 1]
)

### Fix 2 — threshold tuning

In [ ]:
y_prob = weighted_model.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (
    precisions[:-1] + recalls[:-1] + 1e-9
)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Best threshold (max F1): {best_threshold:.3f}  (default was 0.5)")

y_pred_tuned = (y_prob >= best_threshold).astype(int)
evaluate("Fix 2 — class weights + threshold tuning", y_test, y_pred_tuned, y_prob)

### Fix 3 — SMOTE, last resort

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.sum()} defaults / {len(y_train)} total")
print(f"After  SMOTE: {y_resampled.sum()} defaults / {len(y_resampled)} total")

smote_model = GradientBoostingClassifier(random_state=42)
smote_model.fit(X_resampled, y_resampled)
y_prob_smote = smote_model.predict_proba(X_test)[:, 1]

precisions_s, recalls_s, thresholds_s = precision_recall_curve(y_test, y_prob_smote)
f1_s = 2 * (precisions_s[:-1] * recalls_s[:-1]) / (
    precisions_s[:-1] + recalls_s[:-1] + 1e-9
)
best_thresh_smote = thresholds_s[np.argmax(f1_s)]
y_pred_smote = (y_prob_smote >= best_thresh_smote).astype(int)

evaluate("Fix 3 — SMOTE + threshold tuning", y_test, y_pred_smote, y_prob_smote)

### Before vs. after — read this straight off the “Before vs. After” table slide

In [ ]:
models = {
    "Baseline (no fix)":         (baseline.predict(X_test),
                                  baseline.predict_proba(X_test)[:, 1]),
    "Class weights":             (weighted_model.predict(X_test), y_prob),
    "Class weights + threshold": (y_pred_tuned, y_prob),
    "SMOTE + threshold":         (y_pred_smote, y_prob_smote),
}

print(f"\n{'='*62}")
print(f"{'Model':<30} {'Recall':>8} {'Precision':>10} {'F1':>6} {'AUC':>6}")
print(f"{'='*62}")
for name, (preds, probs) in models.items():
    rec = recall_score(y_test, preds)
    pre = precision_score(y_test, preds, zero_division=0)
    f1  = f1_score(y_test, preds, zero_division=0)
    auc = roc_auc_score(y_test, probs)
    print(f"{name:<30} {rec:>8.1%} {pre:>10.1%} {f1:>6.2f} {auc:>6.3f}")
print(f"{'='*62}")

## Part B — Sections 04-06: A Different Question / The Payoff / Unusual != Risky

*Slide cue: switch here right after “24,977 LendingClub borrowers, no target variable”. Say out loud that this is a different, real dataset from Part A.*

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

DATA_PATH  = "lendingclub_sample_25000.csv"   # update path if needed
OUTPUT_DIR = "."

df = pd.read_csv(DATA_PATH, low_memory=False)

FEATURES = ['int_rate', 'dti', 'fico_range_high', 'annual_inc', 'loan_amnt']
TARGET   = 'bad_loan'

df = df[FEATURES + [TARGET]].dropna()
df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
df = df.dropna()
df[TARGET] = df[TARGET].astype(int)

print(f"Dataset: {len(df):,} loans")
print(f"Bad loan rate (set aside): {df[TARGET].mean():.1%}")

# Features only — no target used anywhere below until the retrospective check
X_anom = df[FEATURES].copy()
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_anom)

### Method 1 — Isolation Forest

In [ ]:
iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=42, n_jobs=-1)
iso.fit(X_scaled)

df['iso_score'] = iso.score_samples(X_scaled)
df['iso_flag']  = (iso.predict(X_scaled) == -1).astype(int)

n_iso = df['iso_flag'].sum()
print(f"Isolation Forest — flagged: {n_iso:,} ({n_iso/len(df):.1%})")
print(f"Bad loan rate — flagged:    {df[df['iso_flag']==1][TARGET].mean():.1%}")
print(f"Bad loan rate — not flagged:{df[df['iso_flag']==0][TARGET].mean():.1%}")

### Method 2 — Local Outlier Factor

In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=False, n_jobs=-1)
lof_preds = lof.fit_predict(X_scaled)

df['lof_score'] = lof.negative_outlier_factor_
df['lof_flag']  = (lof_preds == -1).astype(int)

n_lof = df['lof_flag'].sum()
print(f"LOF — flagged: {n_lof:,} ({n_lof/len(df):.1%})")
print(f"Bad loan rate — flagged:    {df[df['lof_flag']==1][TARGET].mean():.1%}")
print(f"Bad loan rate — not flagged:{df[df['lof_flag']==0][TARGET].mean():.1%}")

### Method 3 — One-Class SVM

*Slower than the other two — sampled to 5,000 rows. Kick this cell off, then talk over it while it runs.*

In [ ]:
sample_idx = np.random.RandomState(42).choice(len(X_scaled), 5000, replace=False)
X_sample  = X_scaled[sample_idx]
df_sample = df.iloc[sample_idx].copy()

ocsvm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
ocsvm.fit(X_sample)

df_sample['ocsvm_score'] = ocsvm.score_samples(X_sample)
df_sample['ocsvm_flag']  = (ocsvm.predict(X_sample) == -1).astype(int)

n_ocsvm = df_sample['ocsvm_flag'].sum()
print(f"One-Class SVM (5k sample) — flagged: {n_ocsvm:,} ({n_ocsvm/len(df_sample):.1%})")
print(f"Bad loan rate — flagged:    {df_sample[df_sample['ocsvm_flag']==1][TARGET].mean():.1%}")
print(f"Bad loan rate — not flagged:{df_sample[df_sample['ocsvm_flag']==0][TARGET].mean():.1%}")

### The payoff — when two independent methods agree

*Slide cue: “When Two Independent Methods Agree.” This is the moment the talk turns.*

In [ ]:
df['n_flags'] = df['iso_flag'] + df['lof_flag']

agree_both = df[df['n_flags'] == 2]
agree_one  = df[df['n_flags'] == 1]
agree_none = df[df['n_flags'] == 0]

print(f"Flagged by BOTH:    {len(agree_both):,} ({len(agree_both)/len(df):.1%}) "
      f"— bad loan rate: {agree_both[TARGET].mean():.1%}")
print(f"Flagged by ONE:     {len(agree_one):,}  ({len(agree_one)/len(df):.1%}) "
      f"— bad loan rate: {agree_one[TARGET].mean():.1%}")
print(f"Flagged by NEITHER: {len(agree_none):,} ({len(agree_none)/len(df):.1%}) "
      f"— bad loan rate: {agree_none[TARGET].mean():.1%}")
print(f"Portfolio average:  {df[TARGET].mean():.1%}")

### Ranking and tiering by severity

*Slide cue: “Not All Anomalies Get the Same Response.”*

In [ ]:
n = len(df)
df['iso_rank'] = df['iso_score'].rank(ascending=True)
df['lof_rank'] = df['lof_score'].rank(ascending=True)
df['iso_pct'] = (df['iso_rank'] / n * 100).round(2)
df['lof_pct'] = (df['lof_rank'] / n * 100).round(2)
df['composite_pct']  = ((df['iso_pct'] + df['lof_pct']) / 2).round(2)
df['composite_rank'] = df['composite_pct'].rank(ascending=True).astype(int)

def assign_tier(pct):
    if pct <= 1.0:    return 'Extreme (top 1%)'
    elif pct <= 2.0:  return 'High (top 2%)'
    elif pct <= 5.0:  return 'Moderate (top 5%)'
    elif pct <= 10.0: return 'Borderline (top 10%)'
    else:             return 'Normal'

df['anomaly_tier'] = df['composite_pct'].apply(assign_tier)
print("Borrowers per tier:")
print(df['anomaly_tier'].value_counts())

### The top 20 — unusual != risky

*Slide cue: the “4 of 20” slide. Open the three example borrowers (benign outlier / data quality issue / genuinely risky) directly from the table this cell prints, and narrate them live.*

In [ ]:
output_cols = ['composite_rank', 'anomaly_tier', 'composite_pct',
               'iso_score', 'iso_rank', 'lof_score', 'lof_rank',
               'int_rate', 'dti', 'fico_range_high',
               'annual_inc', 'loan_amnt', TARGET]
top20 = df[output_cols].sort_values('composite_rank').head(20)

print("Top 20 most anomalous borrowers:")
print(top20.to_string(index=False))
print(f"\nOf the top 20, {top20[TARGET].sum()} actually defaulted "
      f"({top20[TARGET].sum()}/20 = {top20[TARGET].mean():.0%}).")